In [1]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])

# Seu código continua abaixo...
import requests
import json

In [3]:
import sys
import subprocess

# Garante que a biblioteca 'requests' está disponível no interpretador correto
try:
    import requests
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

import json

# ==========================================
# CONFIGURAÇÕES DE CONEXÃO DA API
# ==========================================
# Insira seu Token Administrador que funcionou nos testes anteriores
ASTRA_TOKEN = "AstraCS:aPQHYcpxtLEgezUcnDRyaCPF:2c753f0549a83c3a4e0fab7c7fb0e4f49b8a2d3a8696791122864f156c205717" 

BASE_URL = "https://68a5bf7f-bbf0-407b-ab3e-0a8aa6c11a2a-us-east-2.apps.astra.datastax.com"
KEYSPACE = "default_keyspace"  # Altere para "teste_vetorizacao" se o seu banco principal for este
TABLE = "leituras_sensor"

endpoint_url = f"{BASE_URL}/api/rest/v2/keyspaces/{KEYSPACE}/{TABLE}"

headers = {
    "X-Cassandra-Token": ASTRA_TOKEN,
    "Content-Type": "application/json"
}

# ==========================================
# INTERFACE DINÂMICA VIA TERMINAL (INPUTS)
# ==========================================
print("=" * 50)
print("🔍 SISTEMA DE CONSULTA DE TELEMETRIA NOSQL")
print("=" * 50)

# O usuário digita os valores que deseja pesquisar
sensor_escolhido = input("Digite o ID do Sensor (ex: sensor-001, sensor-003): ").strip()
data_escolhida = input("Digite a Data da Leitura (padrão AAAA-MM-DD, ex: 2026-05-22): ").strip()

# Montagem dinâmica dos parâmetros de busca baseada nas entradas do usuário
query_params = {
    "where": json.dumps({
        "sensor_id": {"$eq": sensor_escolhido},
        "data_leitura": {"$eq": data_escolhida}
    }),
    "page-size": 10
}

print("\n📡 Enviando requisição dinâmica para o cluster Cassandra...")
print(f"Buscando por {sensor_escolhido} no dia {data_escolhida}...\n")

try:
    # Executa a chamada GET HTTP passando os parâmetros informados pelo usuário
    response = requests.get(endpoint_url, headers=headers, params=query_params)
    
    if response.status_code == 200:
        dados = response.json()
        registros = dados.get("data", [])
        
        if registros:
            print(f"✅ Sucesso! Foram encontradas {dados.get('count')} leituras.")
            print("\n--- RESULTADO DA FILTRAGEM ---")
            for reg in registros:
                print(f"Horário: {reg.get('horario')} | "
                      f"Temp: {reg.get('temperatura')}°C | "
                      f"Umidade: {reg.get('umidade')}% | "
                      f"Status: {reg.get('status')}")
        else:
            print("⚠️ Nenhuns dados encontrados para este Sensor nesta Data específica.")
            print("Verifique se digitou o nome corretamente ou se o dado está no outro Keyspace.")
            
    elif response.status_code == 401:
        print("❌ Erro de Autenticação: O Token fornecido foi rejeitado.")
    else:
        print(f"❌ Erro na API (Código HTTP {response.status_code}): {response.text}")

except Exception as e:
    print(f"❌ Erro crítico de comunicação: {e}")

print("\n" + "=" * 50)

🔍 SISTEMA DE CONSULTA DE TELEMETRIA NOSQL

📡 Enviando requisição dinâmica para o cluster Cassandra...
Buscando por sensor-004 no dia 2026-06-08...

✅ Sucesso! Foram encontradas 1 leituras.

--- RESULTADO DA FILTRAGEM ---
Horário: 2026-06-08T19:10:42Z | Temp: 25.9°C | Umidade: 61.3% | Status: OK



In [2]:
import sys
import subprocess
from datetime import datetime

# Garante que a biblioteca 'requests' está disponível no interpretador correto
try:
    import requests
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "requests"])
    import requests

import json

# ==========================================
# CONFIGURAÇÕES DE CONEXÃO DA API REST v2
# ==========================================
# Utilize o seu Token Administrador ativo (AstraCS:...)
ASTRA_TOKEN = "AstraCS:hHNTgUzFfjjFCSJiOmHyQiZA:7fdae3e39a2a49c1849cd09b6b1be0f18268340ace2cf61b0214f8e741ab5fd3" 

BASE_URL = "https://68a5bf7f-bbf0-407b-ab3e-0a8aa6c11a2a-us-east-2.apps.astra.datastax.com"
KEYSPACE = "default_keyspace"  # Mude para "teste_vetorizacao" se for o seu keyspace principal
TABLE = "leituras_sensor"

endpoint_url = f"{BASE_URL}/api/rest/v2/keyspaces/{KEYSPACE}/{TABLE}"

headers = {
    "X-Cassandra-Token": ASTRA_TOKEN,
    "Content-Type": "application/json"
}

# ==========================================
# INTERFACE DE LANÇAMENTO VIA TERMINAL
# ==========================================
print("=" * 50)
print("📥 SISTEMA DE LANÇAMENTO DE TELEMETRIA (NOSQL)")
print("=" * 50)

# 1. Captura dos dados dinâmicos do utilizador
sensor_id = input("Introduza o ID do Sensor (ex: sensor-004): ").strip()

try:
    temperatura = float(input("Introduza a Temperatura (°C - ex: 26.8): "))
    umidade = float(input("Introduza a Humidade (% - ex: 55.4): "))
except ValueError:
    print("❌ Erro: Temperatura e Humidade devem ser valores numéricos (use ponto em vez de vírgula).")
    sys.exit()

status = input("Introduza o Status (OK, ALERTA, CRÍTICO): ").strip().upper()

# 2. Geração automática do carimbo de data/hora no padrão ISO 8601 do Cassandra
data_atual = datetime.utcnow().strftime("%Y-%m-%d")
horario_atual = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

# 3. Montagem dinâmica do Payload JSON
novo_registro = {
    "sensor_id": sensor_id,
    "data_leitura": data_atual,
    "horario": horario_atual,
    "temperatura": temperatura,
    "umidade": umidade,
    "status": status
}

print("\n📡 A enviar dados para o cluster Cassandra em us-east-2...")
print(f"A persistir no Keyspace [{KEYSPACE}]...")

try:
    # Executa o pedido HTTP POST para inserir o registo dinâmico
    response = requests.post(endpoint_url, headers=headers, json=novo_registro)
    
    if response.status_code in [200, 201]:
        print("\n✅ SUCESSO! Os dados foram lançados e replicados no Cassandra!")
        print("Dados gravados:")
        print(f" > Chave Primária: ({sensor_id} | {data_atual} | {horario_atual})")
        print(f" > Valores: Temp: {temperatura}°C | Humidade: {umidade}% | Status: {status}")
    else:
        print(f"\n❌ Falha no Lançamento (Código HTTP {response.status_code})")
        print(f"Mensagem do Astra: {response.text}")

except Exception as e:
    print(f"\n❌ Erro crítico de comunicação de rede: {e}")

print("=" * 50)

📥 SISTEMA DE LANÇAMENTO DE TELEMETRIA (NOSQL)


C:\Users\User\AppData\Local\Temp\ipykernel_20652\2076351268.py:51: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  data_atual = datetime.utcnow().strftime("%Y-%m-%d")
C:\Users\User\AppData\Local\Temp\ipykernel_20652\2076351268.py:52: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  horario_atual = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")



📡 A enviar dados para o cluster Cassandra em us-east-2...
A persistir no Keyspace [default_keyspace]...

✅ SUCESSO! Os dados foram lançados e replicados no Cassandra!
Dados gravados:
 > Chave Primária: (sensor-004 | 2026-06-08 | 2026-06-08T19:10:42Z)
 > Valores: Temp: 25.9°C | Humidade: 61.3% | Status: OK
